# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Access the metadata (do not treat as dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}\nLicense: {metadata.license}\nKeywords: {getattr(metadata, 'keywords', None)}")

## 2. Data Overview
Review available record sets (`@id`), fields, and their IDs.

We will list all record sets and, for each, all fields defined by their `@id`. This helps in referencing them programmatically and understanding the dataset's available tables/structures.

In [ ]:
# List all available record sets and their fields by @id
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets:\n")
record_set_ids = []
for record_set in record_sets:
    print(f"- Record Set @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    if 'fields' in record_set and record_set['fields']:
        print("  Fields:")
        for field in record_set['fields']:
            field_id = field['@id']
            print(f"    - {field_id}")
    else:
        print("  (No fields listed)")
    print("")

# Preview the first available record set @id
if record_set_ids:
    preview_record_set_id = record_set_ids[0]
    print(f"Example record set to explore: {preview_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always use the record set and field `@id` values retrieved above for accessing data.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Display the first few rows from the first available record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nPreviewing first 5 rows from record set: {first_rs_id}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Typical exploratory steps: filter records, normalize numeric fields, and group records. For demonstration, we'll attempt these steps for the first record set loaded with available numeric columns.

In [ ]:
# EDA on first loaded record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()
    print(f"Operating on record set: {record_set_id}\nColumns: {df.columns.tolist()}")
    
    # Identify numeric fields automatically for demonstration
    numeric_fields = df.select_dtypes(include="number").columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean()  # Use mean as an arbitrary threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df[[numeric_field]].head())

        # Normalize numeric column
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt to group by a non-numeric field if present
        possible_group_fields = [c for c in df.columns if c not in numeric_fields and df[c].nunique() < len(df)/2]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field, dropna=False)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (showing mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields available for analysis in this record set.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization

Visualize data distributions or relationships. We'll show histograms and, if possible, a boxplot for a numeric field in the explored DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_fields = df.select_dtypes(include="number").columns.tolist()
    if numeric_fields:
        field = numeric_fields[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[field].dropna(), kde=True)
        plt.title(f"Distribution of {field}")
        plt.xlabel(field)
        plt.show()
        
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[field])
        plt.title(f"Boxplot of {field}")
        plt.show()
    else:
        print("No numeric field found for visualization.")
else:
    print("No data loaded for visualization.")

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load and explore the FAIR^2 dataset. We:
- Inspected metadata and listed all record sets and their fields by `@id`.
- Extracted records from available record sets.
- Performed basic exploratory data analysis, including filtering and normalization of a numeric field.
- Produced basic distribution visualizations for a selected numeric variable.

**Next steps** could include deeper analyses on regression results, further subgroup comparisons by socio-demographic fields, or building predictive models using these data tables.

For further exploration, consult the Croissant schema documentation and use the explicit `@id` references identified above.